# Unit 6 Hands-On: Actor-Critic (A2C) — PandaReachDense-v3

이 노트북은 **Hugging Face 딥 강화학습 강좌 Unit 6**의 실습입니다.  
로봇 팔(Panda)이 목표 지점에 손끝을 갖다 대는 작업을  
**A2C(Advantage Actor-Critic)** 알고리즘으로 훈련합니다.

### Unit 4(REINFORCE)와의 차이점

| 항목 | Unit 4 (REINFORCE) | Unit 6 (A2C) |
|---|---|---|
| **알고리즘** | Policy Gradient (MC) | Actor-Critic (TD) |
| **가치 함수** | 없음 | Critic이 V(s) 추정 |
| **업데이트** | 에피소드 종료 후 | 매 스텝마다 |
| **분산** | 높음 | 낮음 (Baseline으로 감소) |
| **구현** | 직접 PyTorch | Stable-Baselines3 |

### A2C 핵심 구조
```
Actor  → 정책 π(a|s) 출력 → 행동 선택
Critic → 상태 가치 V(s) 추정 → Advantage 계산

Advantage A(s,a) = Q(s,a) - V(s)
Loss = -log π(a|s) × A(s,a)  +  Value Loss  +  Entropy Bonus
```

### VecNormalize란?
관측값과 보상을 **평균 0, 표준편차 1로 정규화**하는 래퍼입니다.  
PandaReach는 관측값 스케일이 크므로 정규화 없이는 학습이 불안정합니다.  
훈련 통계(`vec_normalize.pkl`)를 저장하여 평가/추론 시 동일하게 적용합니다.

---
## 목차
1. 환경 설치
2. Google Drive 마운트
3. 가상 디스플레이 설정
4. 라이브러리 임포트
5. 환경 탐색
6. 벡터화 환경 및 VecNormalize 설정
7. A2C 모델 정의 및 훈련
8. 모델 저장
9. 모델 평가
10. 훈련 영상 저장
11. Hugging Face Hub 업로드


---
## 1. 환경 설치

pybullet은 Python 3.12용 사전 빌드 wheel이 없어 소스 빌드가 불가능합니다.  
Miniconda로 **Python 3.10 가상환경**을 구성한 뒤 그 환경의 pip으로 패키지를 설치합니다.

| 셀 | 내용 |
|---|---|
| 1-1 | apt + pyvirtualdisplay 설치 |
| 1-2 | Miniconda 설치 + Python 3.10 conda 환경 생성 |
| 1-3 | conda 환경 Python 버전 확인 |
| 1-4 | conda 환경에 패키지 설치 |


In [ ]:
%%capture
!apt install -y python-opengl ffmpeg xvfb
!pip install pyvirtualdisplay


In [ ]:
import os
import sys
import shutil
import subprocess
import urllib.request

MINICONDA_DIR = '/content/miniconda3'
ENV_NAME      = 'panda_env'

if 'google.colab' in sys.modules:
    candidate_paths = [
        os.path.join(MINICONDA_DIR, 'bin', 'conda'),
        os.path.expanduser('~/miniconda3/bin/conda'),
        '/usr/local/bin/conda',
    ]
    conda_path = next((p for p in candidate_paths if os.path.exists(p)), None)

    if not conda_path:
        print('Colab 환경에서 Miniconda를 설치합니다...')
        installer = '/tmp/miniconda.sh'
        urls = [
            'https://repo.anaconda.com/miniconda/Miniconda3-py310_24.11.3-0-Linux-x86_64.sh',
            'https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh',
        ]

        downloaded = False
        for url in urls:
            try:
                print(f'다운로드 시도: {url}')
                urllib.request.urlretrieve(url, installer)
                downloaded = True
                print('✅ 다운로드 완료')
                break
            except Exception as e:
                print(f'⚠️ 실패: {url} -> {e}')

        if not downloaded:
            raise RuntimeError('Miniconda 설치 파일을 다운로드하지 못했습니다.')

        result = subprocess.run(
            ['bash', installer, '-b', '-p', MINICONDA_DIR],
            capture_output=True, text=True, check=False,
        )
        if result.stdout: print(result.stdout)
        if result.stderr: print(result.stderr)
        conda_path = next((p for p in candidate_paths if os.path.exists(p)), None)

    if not conda_path:
        raise RuntimeError('Miniconda 설치가 완료되지 않았습니다.')

    print('conda 경로:', conda_path)

    # conda 채널 약관 승인 (필수)
    for channel in [
        'https://repo.anaconda.com/pkgs/main',
        'https://repo.anaconda.com/pkgs/r',
    ]:
        subprocess.run(
            [conda_path, 'tos', 'accept', '--override-channels', '--channel', channel],
            capture_output=True, text=True, check=False,
        )

    # Python 3.10.12 conda 환경 생성
    try:
        subprocess.run(
            [conda_path, 'create', '-y', '-n', ENV_NAME, 'python=3.10.12', 'ujson'],
            capture_output=True, text=True, check=True,
        )
        print('✅ conda 환경 생성 완료:', ENV_NAME)
    except subprocess.CalledProcessError as e:
        if 'already exists' in (e.stdout or '') + (e.stderr or ''):
            print('ℹ️ 이미 존재하는 환경입니다. 기존 환경을 사용합니다.')
        else:
            print('❌ conda 환경 생성 실패')
            print(e.stdout); print(e.stderr)
            raise

else:
    print('Colab 환경이 아닙니다.')
    if shutil.which('conda'):
        print(f'로컬 conda로 환경을 생성하세요: conda create -y -n {ENV_NAME} python=3.10.12 ujson')
    else:
        print('Miniconda를 먼저 설치하세요: https://www.anaconda.com/download/success')


In [ ]:
import os
import subprocess

MINICONDA_DIR = '/content/miniconda3'
ENV_NAME      = 'panda_env'
conda_path    = os.path.join(MINICONDA_DIR, 'bin', 'conda')

if os.path.exists(conda_path):
    result = subprocess.run(
        [conda_path, 'run', '-n', ENV_NAME, 'python', '--version'],
        capture_output=True, text=True, check=False,
    )
    print(result.stdout.strip() or result.stderr.strip())
else:
    print(f'conda 실행 파일이 없습니다: {conda_path}')
    print('이전 셀에서 Miniconda 설치가 실패했는지 확인하세요.')


In [ ]:
import os, subprocess

MINICONDA_DIR = '/content/miniconda3'
ENV_NAME      = 'panda_env'
conda_path    = os.path.join(MINICONDA_DIR, 'bin', 'conda')
pip_path      = os.path.join(MINICONDA_DIR, 'envs', ENV_NAME, 'bin', 'pip')

packages = [
    'pybullet',                      # Python 3.10용 wheel 즉시 설치
    'stable-baselines3[extra]',
    'gymnasium',
    'huggingface_sb3',
    'huggingface_hub<1.0',
    'panda_gym',
    'imageio',
    'imageio-ffmpeg',
]

for pkg in packages:
    print(f'설치 중: {pkg}')
    result = subprocess.run(
        [pip_path, 'install', pkg],
        capture_output=True, text=True, check=False,
    )
    if result.returncode == 0:
        print(f'  ✅ {pkg}')
    else:
        print(f'  ❌ {pkg}')
        print(result.stderr[-300:])  # 오류 마지막 300자만 출력

print('\n✅ 모든 패키지 설치 완료')


---
## 2. Google Drive 마운트

Colab VM은 세션 종료 시 파일이 삭제됩니다.  
모델, 정규화 통계, 훈련 영상을 Drive에 저장합니다.

```
Google Drive/RL_Course/Unit6_A2C/
├── a2c-PandaReachDense-v3.zip   ← 훈련된 모델
├── vec_normalize.pkl            ← VecNormalize 통계
└── training_videos/
    └── panda_trained.mp4        ← 평가 영상
```


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ✏️ 저장 경로를 원하는 대로 변경하세요.
DRIVE_BASE = '/content/drive/MyDrive/RL_Course/Unit6_A2C'
VIDEO_DIR  = f'{DRIVE_BASE}/training_videos'
MODEL_DIR  = DRIVE_BASE

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print('✅ Drive 마운트 완료!')
print(f'   모델 저장 경로 : {MODEL_DIR}')
print(f'   영상 저장 경로 : {VIDEO_DIR}')


---
## 3. 가상 디스플레이 설정


In [ ]:
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()
print('✅ 가상 디스플레이 시작')


---
## 4. 라이브러리 임포트

| 라이브러리 | 역할 |
|---|---|
| `panda_gym` | Panda 로봇 팔 gymnasium 환경 |
| `A2C` | Stable-Baselines3의 Actor-Critic 구현 |
| `VecNormalize` | 관측값/보상 실시간 정규화 래퍼 |
| `make_vec_env` | 병렬 환경 생성 유틸리티 |
| `package_to_hub` | SB3 모델을 HF Hub에 업로드 |


In [ ]:
import os, subprocess, sys

MINICONDA_DIR = '/content/miniconda3'
ENV_NAME      = 'panda_env'
conda_python  = os.path.join(MINICONDA_DIR, 'envs', ENV_NAME, 'bin', 'python')

# conda Python이 존재하는지 확인
if not os.path.exists(conda_python):
    raise FileNotFoundError(
        f'conda Python을 찾을 수 없습니다: {conda_python}\n'
        '→ Cell 03~05 (Miniconda 설치 및 패키지 설치) 를 먼저 실행하세요.'
    )

result = subprocess.run([conda_python, '--version'], capture_output=True, text=True)
print('conda Python:', result.stdout.strip())

# conda 환경에서 임포트 테스트
test = subprocess.run(
    [conda_python, '-c',
     'import gymnasium, panda_gym, stable_baselines3;'
     'print("gymnasium:", gymnasium.__version__);'
     'print("stable_baselines3:", stable_baselines3.__version__);'
     'print("panda_gym: OK")'],
    capture_output=True, text=True
)
if test.returncode == 0:
    print(test.stdout.strip())
    print('\n✅ conda 환경 패키지 확인 완료')
else:
    print('❌ 임포트 실패:')
    print(test.stderr[-500:])

# conda_python 경로를 전역 변수로 저장 (이후 셀에서 사용)
os.environ['CONDA_PYTHON'] = conda_python
print(f'\nCONDA_PYTHON 환경변수 설정: {conda_python}')


In [ ]:
# ── 훈련 방식 안내 ───────────────────────────────────────────────
# Colab 커널(Python 3.12)과 conda 환경(Python 3.10)은 서로 다른 인터프리터라
# import 방식으로는 패키지를 공유할 수 없습니다.
#
# 대신 conda Python으로 훈련 스크립트를 직접 실행합니다:
# subprocess.run([conda_python, 'train.py'])
#
# 아래 셀들은 훈련 스크립트(train.py)를 생성하고 conda Python으로 실행합니다.

import os
conda_python = os.environ.get('CONDA_PYTHON', '/content/miniconda3/envs/panda_env/bin/python')
print(f'사용할 Python: {conda_python}')


---
## 5. 환경 탐색

### PandaReachDense-v3 란?

Franka Panda 로봇 팔의 손끝(end-effector)을 목표 위치로 이동시키는 환경입니다.  
`Dense` 보상: 목표까지의 거리에 비례한 **연속적인 보상** (가까울수록 높음)

**관측 공간** (딕셔너리 형태):
- `observation`: 로봇 팔의 관절 상태 (위치, 속도)
- `achieved_goal`: 현재 손끝 위치 (x, y, z)
- `desired_goal`: 목표 위치 (x, y, z)

→ 딕셔너리 관측이므로 정책에 `MultiInputPolicy` 사용

**행동 공간**: 3차원 연속값 (손끝의 x, y, z 방향 이동 속도)


In [ ]:
import subprocess, os

conda_python = os.environ.get('CONDA_PYTHON', '/content/miniconda3/envs/panda_env/bin/python')

env_explore_code = '''
import gymnasium as gym
import panda_gym

env_id = 'PandaReachDense-v3'
env    = gym.make(env_id)

print('===== 관측 공간(Observation Space) =====')
print('관측 공간:', env.observation_space)
print('샘플 관측:', env.observation_space.sample())

print('\\n===== 행동 공간(Action Space) =====')
print('행동 공간:', env.action_space)
print('샘플 행동:', env.action_space.sample())

env.close()
'''

result = subprocess.run([conda_python, '-c', env_explore_code],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('❌ 오류:', result.stderr[-500:])


---
## 6. 벡터화 환경 및 VecNormalize 설정

```
make_vec_env(n_envs=4)     ← 4개 환경 병렬 실행 (데이터 수집 효율화)
     │
VecNormalize               ← 관측값/보상 정규화
  norm_obs=True            ← 관측값을 평균 0, 표준편차 1로 정규화
  norm_reward=True         ← 보상도 정규화 (훈련 안정화)
  clip_obs=10.             ← 관측값을 [-10, 10] 범위로 클리핑
```

> ⚠️ VecNormalize 통계는 훈련 환경에서만 업데이트됩니다.  
> 평가 시에는 `eval_env.training = False`로 고정해야 합니다.


In [ ]:
# 전체 훈련 코드를 train.py로 저장
# conda Python으로 이 파일을 실행합니다.

import os

DRIVE_BASE    = '/content/drive/MyDrive/RL_Course/Unit6_A2C'  # Drive 마운트 경로
MODEL_DIR     = DRIVE_BASE
VIDEO_DIR     = f'{DRIVE_BASE}/training_videos'
MODEL_NAME    = 'a2c-PandaReachDense-v3'
ENV_ID        = 'PandaReachDense-v3'
TOTAL_STEPS   = 1_000_000

train_script = f'''
import os, sys
import numpy as np
import imageio
import gymnasium as gym
import panda_gym

from stable_baselines3 import A2C
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.env_util import make_vec_env

ENV_ID      = "{ENV_ID}"
MODEL_DIR   = "{MODEL_DIR}"
VIDEO_DIR   = "{VIDEO_DIR}"
MODEL_NAME  = "{MODEL_NAME}"
model_path  = os.path.join(MODEL_DIR, MODEL_NAME)
vec_path    = os.path.join(MODEL_DIR, "vec_normalize.pkl")
video_path  = os.path.join(VIDEO_DIR, "panda_trained.mp4")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)

# ── 1. 벡터화 환경 + VecNormalize ────────────────────────────────
env = make_vec_env(ENV_ID, n_envs=4)
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)
print("✅ 환경 생성 완료")

# ── 2. A2C 모델 훈련 ─────────────────────────────────────────────
model = A2C(policy="MultiInputPolicy", env=env, verbose=1)
model.learn(total_timesteps={TOTAL_STEPS})
print("\\n✅ 훈련 완료!")

# ── 3. 모델 및 VecNormalize 통계 저장 ────────────────────────────
model.save(model_path)
env.save(vec_path)
print(f"✅ 모델 저장: {{model_path}}.zip")
print(f"✅ VecNormalize 저장: {{vec_path}}")

# ── 4. 평가 ──────────────────────────────────────────────────────
eval_env = DummyVecEnv([lambda: gym.make(ENV_ID)])
eval_env = VecNormalize.load(vec_path, eval_env)
eval_env.training    = False
eval_env.norm_reward = False
model = A2C.load(model_path, env=eval_env)
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"평균 보상: {{mean_reward:.2f}} +/- {{std_reward:.2f}}")

# ── 5. 영상 저장 ─────────────────────────────────────────────────
record_env = DummyVecEnv([lambda: gym.make(ENV_ID, render_mode="rgb_array")])
record_env = VecNormalize.load(vec_path, record_env)
record_env.training    = False
record_env.norm_reward = False

frames = []
obs = record_env.reset()
for _ in range(500):
    action, _ = model.predict(obs, deterministic=True)
    obs, _, dones, _ = record_env.step(action)
    frame = record_env.get_images()
    if frame[0] is not None:
        frames.append(frame[0])
    if dones[0]:
        break

record_env.close()
imageio.mimsave(video_path, [np.array(f) for f in frames], fps=30)
print(f"✅ 영상 저장 완료: {{video_path}}")
print("\\n모든 작업 완료!")
'''

with open('/content/train.py', 'w') as f:
    f.write(train_script)

print('✅ train.py 생성 완료: /content/train.py')
print(f'   환경: {ENV_ID}')
print(f'   훈련 스텝: {TOTAL_STEPS:,}')
print(f'   모델 저장: {MODEL_DIR}')
print(f'   영상 저장: {VIDEO_DIR}')


---
## 7. 훈련 실행

위에서 생성한 `train.py`를 **conda Python(3.10)으로 직접 실행**합니다.  
훈련 → 평가 → 영상 저장까지 한 번에 처리합니다.

> ⏱️ 예상 시간: **약 20~40분** (n_envs=4, 100만 스텝)  
> 훈련 로그가 실시간으로 출력됩니다.


In [ ]:
import subprocess, os

conda_python = os.environ.get('CONDA_PYTHON', '/content/miniconda3/envs/panda_env/bin/python')

print(f'실행: {conda_python} /content/train.py')
print('=' * 50)

# conda Python으로 훈련 스크립트 실행
# 훈련 로그를 실시간으로 출력
process = subprocess.Popen(
    [conda_python, '/content/train.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()

if process.returncode == 0:
    print('\n✅ 훈련 완료!')
else:
    print(f'\n❌ 오류 발생 (returncode={process.returncode})')


In [ ]:
# 저장된 영상을 노트북에서 바로 확인
from IPython.display import Video, display
import os

DRIVE_BASE = '/content/drive/MyDrive/RL_Course/Unit6_A2C'
video_path = f'{DRIVE_BASE}/training_videos/panda_trained.mp4'

if os.path.exists(video_path):
    display(Video(video_path, embed=True, width=500))
else:
    print(f'영상 파일이 없습니다: {video_path}')
    print('훈련 셀을 먼저 실행하세요.')


---
## 8. 모델 저장 / 9. 평가 / 10. 영상 저장

훈련 스크립트(`train.py`)에서 자동으로 처리합니다:

| 작업 | 저장 경로 |
|---|---|
| 모델 | `Google Drive/RL_Course/Unit6_A2C/a2c-PandaReachDense-v3.zip` |
| VecNormalize 통계 | `Google Drive/RL_Course/Unit6_A2C/vec_normalize.pkl` |
| 평가 영상 | `Google Drive/RL_Course/Unit6_A2C/training_videos/panda_trained.mp4` |


In [ ]:
# train.py에서 자동 처리됨 (실행 불필요)


---
## 9. 모델 평가

평가 시 VecNormalize 설정:
- `training = False`: 정규화 통계를 업데이트하지 않음 (고정)
- `norm_reward = False`: 평가 시 보상 정규화 불필요 (실제 보상 확인)


In [ ]:
# train.py에서 자동 처리됨 (실행 불필요)


---
## 10. 훈련 영상 저장

평가용 환경으로 에피소드를 실행하면서 프레임을 수집하여 mp4로 저장합니다.  

> ⚠️ VSCode + Colab 환경에서는 `render_mode='human'`이 커널을 죽이므로  
> `rgb_array` 모드로 프레임을 수집합니다.


In [ ]:
# train.py에서 자동 처리됨 (실행 불필요)


---
## 11. Hugging Face Hub 업로드

`package_to_hub`가 아래 작업을 자동으로 처리합니다:
- 모델 평가 → 결과 기록
- 플레이 영상 자동 생성
- 모델 카드 작성
- HF Hub 업로드

### 사전 준비
1. [HF 계정 생성](https://huggingface.co/join)
2. [Write 토큰 발급](https://huggingface.co/settings/tokens)


In [ ]:
# ✏️ 본인의 HF 토큰 입력
HF_TOKEN = 'hf_xxxxxxxxxxxxxxxxxxxxxxxx'

import subprocess, os
conda_python = os.environ.get('CONDA_PYTHON', '/content/miniconda3/envs/panda_env/bin/python')

# conda 환경에서 HF 로그인
subprocess.run(
    [conda_python, '-c', f'from huggingface_hub import login; login(token="{HF_TOKEN}")'],
    check=False
)
!git config --global credential.helper store
print('✅ HF 로그인 완료')


In [ ]:
import subprocess, os

conda_python  = os.environ.get('CONDA_PYTHON', '/content/miniconda3/envs/panda_env/bin/python')
DRIVE_BASE    = '/content/drive/MyDrive/RL_Course/Unit6_A2C'
ENV_ID        = 'PandaReachDense-v3'
MODEL_NAME    = 'a2c-PandaReachDense-v3'
HF_USERNAME   = 'YOUR_HF_USERNAME'  # ✏️ 본인 HF 사용자명으로 변경
HF_TOKEN      = 'hf_xxxxxxxxxxxxxxxxxxxxxxxx'  # ✏️ 본인 토큰

hub_script = f'''
import os
import gymnasium as gym
import panda_gym
from huggingface_hub import login
from huggingface_sb3 import package_to_hub
from stable_baselines3 import A2C
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

login(token="{HF_TOKEN}")

ENV_ID     = "{ENV_ID}"
MODEL_DIR  = "{DRIVE_BASE}"
model_path = os.path.join(MODEL_DIR, "{MODEL_NAME}")
vec_path   = os.path.join(MODEL_DIR, "vec_normalize.pkl")

eval_env = DummyVecEnv([lambda: gym.make(ENV_ID)])
eval_env = VecNormalize.load(vec_path, eval_env)
eval_env.training    = False
eval_env.norm_reward = False

model = A2C.load(model_path, env=eval_env)

package_to_hub(
    model=model,
    model_name="{MODEL_NAME}",
    model_architecture="A2C",
    env_id=ENV_ID,
    eval_env=eval_env,
    repo_id="{HF_USERNAME}/a2c-{ENV_ID}",
    commit_message="Initial commit",
)
print("✅ HF Hub 업로드 완료")
'''

with open('/content/push_hub.py', 'w') as f:
    f.write(hub_script)

process = subprocess.Popen(
    [conda_python, '/content/push_hub.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode == 0:
    print(f'\n✅ 업로드 완료: https://huggingface.co/{HF_USERNAME}/a2c-{ENV_ID}')
else:
    print(f'\n❌ 업로드 실패 (returncode={process.returncode})')
